In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/housing-prices-prediction-spring-2026/sample_submission.csv
/kaggle/input/housing-prices-prediction-spring-2026/train.csv
/kaggle/input/housing-prices-prediction-spring-2026/test.csv


## Loading Training Data

In [3]:
df_traindata = pd.read_csv('/kaggle/input/housing-prices-prediction-spring-2026/train.csv', low_memory=False)
df_traindata.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,126959
1,2,20,RM,60.0,7200,Pave,Grvl,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2006,COD,Normal,98679
2,3,20,RL,94.0,13615,Pave,NaN,IR1,HLS,AllPub,...,0,NaN,NaN,NaN,0,6,2007,WD,Normal,332371
3,4,20,RL,65.0,14753,Pave,NaN,IR2,Low,AllPub,...,0,NaN,GdPrv,NaN,0,12,2009,WD,Normal,203992
4,5,160,RL,36.0,2448,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,10,2008,WD,Normal,170016


## Data Cleaning

To begin preprocessing, I first examined the structure of the dataset using `.info()` and inspected missing values with `.isna().sum()`. This provided an overview of the data types and helped identify which columns contained missing values.

Several columns had extremely high missingness (over 1,000 NaN values). These columns were removed because the amount of missing data made them unreliable for modeling. I also removed `GarageYrBlt`, since it is almost always the same as `YearBuilt`, which introduces redundancy and strong correlation. In addition, I removed `GarageArea` because it showed a very strong correlation with `GarageCars`, meaning both features were capturing nearly the same information. Keeping both would add unnecessary redundancy without improving model performance.

Next, I focused on the categorical (object-type) features. For each categorical column with missing values, I printed the unique values to understand what the NaNs represented. According to the dataset documentation, many of these NaNs indicate that a feature is *not present* in the home (e.g., no basement, no alley access, no fireplace, etc.).

For these specific categorical features, I replaced NaN with `"None"` to explicitly encode the absence of the feature rather than treating it as an unknown value. This preserves meaningful information for the model and avoids incorrect assumptions during imputation.


In [4]:
df_traindata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   object 
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   Alley          107 non-null    object 
 7   LotShape       1459 non-null   object 
 8   LandContour    1459 non-null   object 
 9   Utilities      1457 non-null   object 
 10  LotConfig      1459 non-null   object 
 11  LandSlope      1459 non-null   object 
 12  Neighborhood   1459 non-null   object 
 13  Condition1     1459 non-null   object 
 14  Condition2     1459 non-null   object 
 15  BldgType       1459 non-null   object 
 16  HouseStyle     1459 non-null   object 
 17  OverallQual    1459 non-null   int64  
 18  OverallC

In [5]:
pd.set_option('display.max_rows', None)
df_traindata.isna().sum()[df_traindata.isna().sum()>0].sort_values(ascending=False)

PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageQual        78
GarageCond        78
GarageYrBlt       78
GarageFinish      78
GarageType        76
BsmtCond          45
BsmtQual          44
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
Functional         2
BsmtFullBath       2
Utilities          2
BsmtHalfBath       2
Exterior1st        1
Exterior2nd        1
TotalBsmtSF        1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
KitchenQual        1
GarageArea         1
GarageCars         1
SaleType           1
dtype: int64

In [6]:
object_cols = df_traindata.select_dtypes(include='object').columns
null_obj_cols = [col for col in object_cols if df_traindata[col].isna().sum() > 0]

for col in null_obj_cols:
    print(f"\nColumn: {col}")
    print(df_traindata[col].unique())



Column: MSZoning
['RH' 'RM' 'RL' 'FV' 'C (all)' nan]

Column: Alley
[nan 'Grvl' 'Pave']

Column: Utilities
['AllPub' nan]

Column: Exterior1st
['VinylSd' 'MetalSd' 'Wd Sdng' 'HdBoard' 'BrkFace' 'CemntBd' 'Plywood'
 'WdShing' 'BrkComm' 'Stucco' 'AsbShng' 'CBlock' 'AsphShn' nan]

Column: Exterior2nd
['VinylSd' 'MetalSd' 'Wd Shng' 'Wd Sdng' 'HdBoard' 'ImStucc' 'BrkFace'
 'CmentBd' 'Brk Cmn' 'Plywood' 'Stucco' 'AsbShng' 'AsphShn' 'CBlock' nan
 'Stone']

Column: MasVnrType
[nan 'Stone' 'BrkFace' 'BrkCmn']

Column: BsmtQual
['TA' 'Ex' 'Gd' 'Fa' nan]

Column: BsmtCond
['TA' 'Gd' 'Fa' nan 'Po']

Column: BsmtExposure
['No' 'Gd' 'Mn' 'Av' nan]

Column: BsmtFinType1
['Rec' 'Unf' 'GLQ' 'BLQ' 'LwQ' 'ALQ' nan]

Column: BsmtFinType2
['LwQ' 'Unf' 'BLQ' 'GLQ' 'Rec' 'ALQ' nan]

Column: KitchenQual
['TA' 'Ex' 'Gd' 'Fa' nan]

Column: Functional
['Typ' 'Min1' 'Mod' 'Min2' 'Maj1' 'Maj2' 'Sev' nan]

Column: FireplaceQu
[nan 'Gd' 'TA' 'Fa' 'Po' 'Ex']

Column: GarageType
['Attchd' 'Detchd' 'BuiltIn' nan 'Basm

In [7]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'GarageYrBlt', 'GarageArea']

df_clean = df_traindata.drop(columns=cols_to_drop)

df_clean.isna().sum()[df_clean.isna().sum()>0].sort_values(ascending=False)

MasVnrType      894
FireplaceQu     730
LotFrontage     227
GarageCond       78
GarageFinish     78
GarageQual       78
GarageType       76
BsmtCond         45
BsmtQual         44
BsmtExposure     44
BsmtFinType1     42
BsmtFinType2     42
MasVnrArea       15
MSZoning          4
Utilities         2
Functional        2
BsmtFullBath      2
BsmtHalfBath      2
Exterior1st       1
Exterior2nd       1
KitchenQual       1
BsmtFinSF2        1
BsmtUnfSF         1
BsmtFinSF1        1
TotalBsmtSF       1
GarageCars        1
SaleType          1
dtype: int64

In [8]:
cols_none = [
    "MasVnrType",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "GarageFinish",
    "GarageType",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2"
]


for col in cols_none:
    df_clean[col] = df_clean[col].fillna("None")

df_clean.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,Inside,...,0,0,120,0,0,6,2010,WD,Normal,126959
1,2,20,RM,60.0,7200,Pave,Reg,Lvl,AllPub,Inside,...,0,0,115,0,0,8,2006,COD,Normal,98679
2,3,20,RL,94.0,13615,Pave,IR1,HLS,AllPub,Corner,...,0,0,0,0,0,6,2007,WD,Normal,332371
3,4,20,RL,65.0,14753,Pave,IR2,Low,AllPub,Inside,...,0,0,0,0,0,12,2009,WD,Normal,203992
4,5,160,RL,36.0,2448,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,10,2008,WD,Normal,170016


## Cleaning Numeric Columns

Next, I examined the numeric features that contained missing values. For the columns in this list, the NaN values typically indicate that the feature does not exist for that particular house (for example, no basement or no garage), or the value was simply left blank in the dataset. In these cases, the most meaningful placeholder is `0`, since it clearly represents “feature not present” rather than an unknown or missing measurement. Replacing these NaNs with 0 ensures the model receives consistent numeric input without introducing unnecessary bias.



In [9]:
cols_zero = ["MasVnrArea", "BsmtHalfBath", "BsmtFullBath",
    "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFinSF1", "GarageCars"]    
for col in cols_zero:
    df_clean[col] = df_clean[col].fillna(0)

## Imputing LotFrontage Using Neighborhood Medians

The `LotFrontage` column contains missing values that represent true missing measurements rather than the absence of a feature. Because lot size varies meaningfully by neighborhood, I imputed these missing values using the median `LotFrontage` within each neighborhood. This preserves important structure in the data by allowing homes in similar neighborhoods to share similar frontage estimates.

This step is performed before splitting the dataset, consistent with the rest of the structural cleaning process. The imputation uses only information already present in the raw training data and does not involve the target variable, so it does not introduce data leakage. After filling missing values using neighborhood-specific medians, any remaining NaNs are replaced with the overall median as a fallback. A global median is used as a fallback to ensure that any neighborhoods with too few observations still receive a reasonable, non missing value, preventing errors during modeling.


In [10]:
lotfrontage_medians = df_clean.groupby('Neighborhood')['LotFrontage'].median()
df_clean['LotFrontage'] = df_clean['LotFrontage'].fillna(df_clean['Neighborhood'].map(lotfrontage_medians))

df_clean['LotFrontage'] = df_clean['LotFrontage'].fillna(df_clean['LotFrontage'].median())

## Feature Engineering
To help the model capture more meaningful patterns in the housing data, I created several engineered features that combine or transform existing columns into more informative representations. These particular columns were chosen because they describe core structural characteristics of a home such as size, age, bathrooms, remodeling history, and outdoor space, which are known to have strong relationships with housing prices.
- **TotalSF** combines above‑ground and basement square footage into a single measure of total livable space. Many models perform better when related square‑footage features are consolidated into one metric.
- **TotalBath** converts full and half bathrooms (including basement bathrooms) into a single numeric value that reflects total bathroom capacity. Half baths are weighted as 0.5 to represent their reduced utility.
- **AgeAtSale** measures how old the home is at the time of sale.
- **Remodeled** is a binary indicator showing whether the home was remodeled at any point after it was built. This captures renovation effects that may influence sale price.
- **PorchSF** aggregates all porch‑related square footage into one feature, giving the model a clearer sense of total outdoor porch space rather than splitting it across multiple smaller columns.
These engineered features provide the model with more interpretable and consolidated information, often improving predictive performance by reducing noise and highlighting meaningful structure in the data.



In [11]:
df_clean['TotalSF'] = df_clean['1stFlrSF'] + df_clean['2ndFlrSF'] + df_clean['TotalBsmtSF']

df_clean['TotalBath'] = (
    df_clean['FullBath'] +
    0.5 * df_clean['HalfBath'] +
    df_clean['BsmtFullBath'] +
    0.5 * df_clean['BsmtHalfBath']
)

df_clean['AgeAtSale'] = df_clean['YrSold'] - df_clean['YearBuilt']

df_clean['Remodeled'] = (df_clean['YearRemodAdd'] != df_clean['YearBuilt']).astype(int)

df_clean['PorchSF'] = (
    df_clean['OpenPorchSF'] +
    df_clean['EnclosedPorch'] +
    df_clean['3SsnPorch'] +
    df_clean['ScreenPorch']
)


## Splitting the Data into Features and Target

Now that the major cleaning steps and feature engineering are done, I can split the dataset into the parts the model will use. At this point, the only things left to impute are the “real” missing values that need statistics like mode, and those will be handled after the split so they’re learned from the training data only. That keeps the validation set clean and avoids any leakage.

Here I separate the dataset into features (X) and the target (y). `SalePrice` is what I’m trying to predict, so it becomes the target variable. I also drop the `Id` column from the features since it’s just an identifier and doesn’t help the model learn anything useful.



In [12]:
X = df_clean.drop(columns=['SalePrice', 'Id'])
y = df_clean['SalePrice']

## Train–Validation Split

I split the dataset into training and validation sets using an 80/20 ratio. 
A fixed `random_state` ensures the split is reproducible. I did not use the 
`stratify` parameter here because the target variable (`SalePrice`) is continuous, 
and stratification only applies to classification problems with discrete class labels.

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Filling Remaining Categorical NaN Values with Mode

There are seven categorical features with a small number of true missing values left in the dataset. Since these columns represent meaningful categories and the amount of missing data is very small, I fill the remaining NaNs using the mode (the most frequent value) for each column.

I created a list of these columns and used `SimpleImputer(strategy='most_frequent')` to learn the mode from the training data only. The imputer is fitted on `X_train` and then applied to both `X_train` and `X_val` to avoid data leakage and keep the validation set independent.


In [14]:
from sklearn.impute import SimpleImputer

mode_cols = ['MSZoning','Utilities','Functional','Exterior2nd',
            'Exterior1st','KitchenQual','SaleType']

mode_imputer = SimpleImputer(strategy='most_frequent')
X_train[mode_cols] = mode_imputer.fit_transform(X_train[mode_cols])
X_val[mode_cols] = mode_imputer.transform(X_val[mode_cols])


## One‑Hot Encoding Categorical Features

After handling the remaining missing values, I convert the categorical columns into dummy variables using `pd.get_dummies()`. This turns each category into its own binary indicator so the model can work with them numerically. I use `drop_first=True` to avoid multicollinearity by removing one level from each categorical feature.

Because the training and validation sets may not contain the exact same categories, I reindex the validation set to match the training columns and fill any missing dummy columns with 0. This keeps both datasets aligned and ensures the model receives the same feature structure during training and evaluation. After encoding, the dataset contains a mix of `float64`, `int64`, and `bool` data types.


In [15]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_train.dtypes

MSSubClass                 int64
LotFrontage              float64
LotArea                    int64
OverallQual                int64
OverallCond                int64
YearBuilt                  int64
YearRemodAdd               int64
MasVnrArea               float64
BsmtFinSF1               float64
BsmtFinSF2               float64
BsmtUnfSF                float64
TotalBsmtSF              float64
1stFlrSF                   int64
2ndFlrSF                   int64
LowQualFinSF               int64
GrLivArea                  int64
BsmtFullBath             float64
BsmtHalfBath             float64
FullBath                   int64
HalfBath                   int64
BedroomAbvGr               int64
KitchenAbvGr               int64
TotRmsAbvGrd               int64
Fireplaces                 int64
GarageCars               float64
WoodDeckSF                 int64
OpenPorchSF                int64
EnclosedPorch              int64
3SsnPorch                  int64
ScreenPorch                int64
PoolArea  

## Feature Scaling with StandardScaler

After one‑hot encoding the categorical features, I standardize the numeric columns using `StandardScaler`. Scaling puts all numeric features on a similar scale, especially when they originally span very different ranges. The scaler is fit only on the training data so the mean and standard deviation come strictly from `X_train`, which prevents data leakage. Those same learned values are then applied to the validation set to keep both datasets on the same scale.

Only the numeric columns are scaled, dummy variables created during one‑hot encoding are left as‑is because they are already binary indicators and do not benefit from standardization.



In [16]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

## Linear Regression
A baseline model that assumes a linear relationship between the features and the target variable. This model provides a useful point of comparison for more complex methods.

In [17]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

## Decision Tree Regressor
A non-linear model capable of capturing feature interactions and threshold base relationships. The random_state parameter is set for reproducibility.

In [18]:
from sklearn.tree import DecisionTreeRegressor

dtr_model = DecisionTreeRegressor(random_state=42)

## Random Forest Regressor
An ensemble of decision trees that reduces overfitting and improves predictive stability. I initialized a Random Forest model with a fixed random state for reproducibility.

In [19]:
from sklearn.ensemble import RandomForestRegressor

rfr_model = RandomForestRegressor(random_state=42)

## K‑Fold Cross‑Validation

To get a more reliable estimate of how well the model generalizes, I use k‑fold cross‑validation on the training data. This technique splits the data into *k* subsets (folds) and trains the model *k* times, each time using a different fold as the validation set and the remaining folds for training. This gives a more stable performance estimate than relying on a single train/validation split and helps reduce the variance that comes from one particular split of the data.

I used 10 folds and evaluated the model with the R² metric. The scores across the folds were all very consistent, averaging around 0.97. This indicates that the model is fitting the training data well without showing signs of instability or overfitting across different subsets. With this level of consistency, it makes sense to move forward with hyperparameter tuning to see if performance can be improved further.



In [20]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rfr_model, X_train, y_train, cv=10, scoring='r2')
scores.mean(), scores


(np.float64(0.9695887866532582),
 array([0.96583059, 0.96778714, 0.95966648, 0.97981977, 0.97021584,
        0.97082019, 0.9629595 , 0.97636173, 0.9720561 , 0.97037052]))

## Hyperparameter Tuning with RandomizedSearchCV
To improve the baseline Random Forest model, I performed hyperparameter tuning using `RandomizedSearchCV`. This method samples combinations of hyperparameters from a predefined search space and evaluates each one using 5 fold cross validation. Randomized search is more efficient than grid search because it explores a wide range of values without testing every possible combination. These hyperparameters were chosen because they allow the model to explore both simpler and more flexible structures. By searching across reasonable ranges for depth, number of trees, feature sampling, and split criteria, `RandomizedSearchCV` can identify the combination that generalizes best.



In [21]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', None]
}


rfr_search = RandomizedSearchCV(
    estimator=rfr_model,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

rfr_search.fit(X_train, y_train)

print("Best parameters:", rfr_search.best_params_)
print("Best CV R²:", rfr_search.best_score_)


Best parameters: {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 10}
Best CV R²: 0.9695539855738392


## Cross‑Validation Results for the Tuned Model

I evaluated the tuned Random Forest model using 10‑fold cross‑validation to check whether the improved hyperparameters continued to generalize well. The R² scores across the folds were all very consistent, with an average around 0.97 and very low variance. This stability suggests that the tuned model is performing reliably across different subsets of the training data and is not overfitting during the tuning process.


In [22]:
best_rfr = rfr_search.best_estimator_
cross_val_score(best_rfr, X_train, y_train, cv=10, scoring='r2')
scores.mean()


array([0.9647432 , 0.96726197, 0.95939171, 0.97887391, 0.96876883,
       0.97081581, 0.96334424, 0.97650122, 0.97083509, 0.96943574])

## Training the Models with the Training Set

Each model was trained using the prepared training data. Linear Regression was fit using the scaled features since it is sensitive to differences in feature magnitude. The Decision Tree and Random Forest models were trained on the unscaled features because tree‑based methods rely on threshold splits and are not affected by feature scaling. Training each model on the appropriate version of the dataset ensures that the preprocessing aligns with how the model learns.


In [23]:
lr_model.fit(X_train_scaled, y_train)

LinearRegression()

In [24]:
dtr_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [25]:
best_rfr.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, max_features=None, n_estimators=800,
                      random_state=42)

## Predicting Labels with Validation Set Features
Each trained model was used to generate predictions on the validation set. Linear Regression predictions were made using the scaled validation features. Decision Tree and Random Forest predictions were made using the unscaled validation data, as tree-based models rely on threshold splits.

In [26]:
# Linear Regression predictions
lr_pred = lr_model.predict(X_val_scaled)

# Decision Tree Regressor predictions
dtr_pred = dtr_model.predict(X_val)

# Random Forest Regressor predictions
rfr_pred = best_rfr.predict(X_val)

## Evaluating Model Performance with R² Score

Each model was evaluated on the validation set using the R² score, which measures how well the model explains the variance in the target variable. Higher values indicate better predictive performance, with 1.0 representing a perfect fit.

The Random Forest achieved the strongest performance on the validation set. The Linear Regression score is still relatively high, but because linear models are more sensitive to the specific train/validation split, its performance can fluctuate more compared to tree‑based models. The tree‑based models show more stable behavior on this dataset, which aligns with their ability to capture nonlinear relationships in the features.


In [27]:
from sklearn.metrics import r2_score

# Linear Regression r2
lr_r2 = r2_score(y_val, lr_pred)

# Decision Tree Regressor r2
dtr_r2 = r2_score(y_val, dtr_pred)

# Random Forest Regressor r2
rfr_r2 = r2_score(y_val, rfr_pred)


print(f"Linear Regression R²: {lr_r2:.3f}")
print(f"Decision Tree R²: {dtr_r2:.3f}")
print(f"Random Forest R²: {rfr_r2:.3f}")

Linear Regression R²: 0.970
Decision Tree R²: 0.945
Random Forest R²: 0.978


## Loading and Preprocessing the Test Data
The hidden test dataset is loaded and prepared for prediction. Since the test set does not include the target variable, only the feature columns are retained. The id column is removed to match the structure of the training data

In [28]:
df_test = pd.read_csv('/kaggle/input/housing-prices-prediction-spring-2026/test.csv', low_memory=False)
df_test_clean = df_test.drop(columns=['Id'])
df_test_clean.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


## Feature Engineering for the Test Set

The test dataset is processed using the exact same steps applied to the training data to keep the feature structure consistent and avoid any form of data leakage. This includes dropping columns, applying the previously fitted imputers to fill missing values, generating the same one‑hot encoded dummy variables, and aligning the test columns with the training set. Numeric features are then scaled using the parameters learned from the training data. These steps ensure that the model receives the test features in the same format and scale as during training.


In [29]:
df_test_clean = df_test.drop(columns=cols_to_drop)

In [30]:
for col in cols_none:
    df_test_clean[col] = df_test_clean[col].fillna("None")

In [31]:
for col in cols_zero:
    df_test_clean[col] = df_test_clean[col].fillna(0)

In [32]:
df_test_clean['LotFrontage'] = df_test_clean['LotFrontage'].fillna(
    df_test_clean['Neighborhood'].map(lotfrontage_medians)
)

df_test_clean['LotFrontage'] = df_test_clean['LotFrontage'].fillna(
    df_test_clean['LotFrontage'].median()
)

In [33]:
df_test_clean['TotalSF'] = (
    df_test_clean['1stFlrSF'] +
    df_test_clean['2ndFlrSF'] +
    df_test_clean['TotalBsmtSF']
)

df_test_clean['TotalBath'] = (
    df_test_clean['FullBath'] +
    0.5 * df_test_clean['HalfBath'] +
    df_test_clean['BsmtFullBath'] +
    0.5 * df_test_clean['BsmtHalfBath']
)

df_test_clean['AgeAtSale'] = df_test_clean['YrSold'] - df_test_clean['YearBuilt']

df_test_clean['Remodeled'] = (
    df_test_clean['YearRemodAdd'] != df_test_clean['YearBuilt']
).astype(int)

df_test_clean['PorchSF'] = (
    df_test_clean['OpenPorchSF'] +
    df_test_clean['EnclosedPorch'] +
    df_test_clean['3SsnPorch'] +
    df_test_clean['ScreenPorch']
)



In [34]:
df_test_clean[mode_cols] = mode_imputer.transform(df_test_clean[mode_cols])

In [35]:
test_features = pd.get_dummies(df_test_clean, drop_first=True)

test_features = test_features.reindex(columns=X_train.columns, fill_value=0)

## Generating Final Predictions on the Hidden Test Set

With the test data fully preprocessed and aligned to the same feature structure as the training set, the tuned Random Forest model is used to generate the final predictions. Because the model was trained on the unscaled feature set, the predictions are made using the processed but unscaled test features. These predicted values will be used to create the submission file for evaluation.



In [36]:
test_pred = best_rfr.predict(test_features)

## Creating benchmark dataframe and saving to a csv file

In [37]:
df_submission = pd.DataFrame({'Id': df_test['Id'], 'SalePrice': test_pred})
df_submission.to_csv('submission.csv', index=False)
df_submission.head()

,Id,SalePrice
0,1,197823.935179
1,2,166312.868218
2,3,210454.514153
3,4,172803.901989
4,5,267378.752049
